# Import sqlalchemy

In [48]:
# Install commands if neede
# pip install SQLAlchemy
# pip install psycopg2

import sqlalchemy as db
from sqlalchemy import text
import pandas as pd
import psycopg2

# Create connection

Connection to local PostgreSQL database

In [49]:
user = 'postgres'
password = 'password0134!'
host = 'localhost'
port = 5432
database = 'db_1'

engine = db.create_engine('postgresql+psycopg2://{0}:{1}@{2}:{3}/{4}'.format(user,
                                                                    password,
                                                                    host,
                                                                    port,
                                                                    database))
conn = engine.connect() 

Querying table to test connection

In [ ]:
result = conn.execute(text("SELECT * FROM course_codes"))

# print all results
for row in result:
    print(row)
    



('10', 'TU099', 'A very good course', True)
('11', 'TU098', 'A very not bad course', False)
('12', 'TU097', 'A very maybe alright  course', True)
('13', 'TU096', 'A very terrible course', False)


In [41]:
result = conn.execute(text("SELECT * FROM dept")).fetchall()
for row in result:
    print(row)

(Decimal('10'), 'ACCOUNTING', 'NEW YORK')
(Decimal('20'), 'RESEARCH', 'DALLAS')
(Decimal('30'), 'SALES', 'CHICAGO')
(Decimal('40'), 'OPERATIONS', 'BOSTON')


In [32]:
result = conn.execute(text("SELECT * FROM student")).fetchmany(3)

# print top 3 results
for row in result:
    print(row)

('D24127620', 'eamonn', 'kelly', datetime.date(1974, 9, 18), 'TU061', '10')
('D020150122', 'Patricia', 'Wilson', datetime.date(1973, 10, 4), 'TU256', '11')
('D020150121', 'John', 'Brown', datetime.date(1987, 9, 18), 'TU256', '12')


In [33]:
result = conn.execute(text("SELECT * FROM student")).fetchall()
data = pd.DataFrame(result)
data

,student_number,first_name,surname,dob,prog_code,course_id
0,D24127620,eamonn,kelly,1974-09-18,TU061,10
1,D020150122,Patricia,Wilson,1973-10-04,TU256,11
2,D020150121,John,Brown,1987-09-18,TU256,12
3,D020150120,James,Smith,1995-01-19,TU256,13


In [46]:
result = conn.execute(text("SELECT * FROM course_codes WHERE course_desc LIKE 'A%%'"))

# print top 3 results
for row in result:
    print(row)

('10', 'TU099', 'A very good course', True)
('11', 'TU098', 'A very not bad course', False)
('12', 'TU097', 'A very maybe alright  course', True)
('13', 'TU096', 'A very terrible course', False)


In [ ]:
result = conn.execute(text("SELECT * FROM course_codes")).fetchall()

data = pd.DataFrame(result)
data

# Show table names

In [ ]:
import sqlalchemy as db
metadata_obj = db.MetaData()
metadata_obj.reflect(bind=engine)
for t in metadata_obj.sorted_tables:
    print(t)

# Reflect tables

In [47]:
# Reflect census table from the engine: census
student = db.Table('student', metadata_obj, autoload=True, autoload_with=engine)

# Print student column names
print(student.columns.keys())

# Print census table metadata
print(repr(student))

NameError: name 'metadata_obj' is not defined

In [ ]:
student = db.Table('student', metadata_obj, autoload=True, autoload_with=engine)
stmt = db.select(student) # Select statement in SQLAlchemy
print(stmt)
print()
# Execute the statement on connection and fetch 10 records: results
results = conn.execute(stmt).fetchall()
print(results)

In [13]:
stmt = db.select(student) # Create a select query: stmt
stmt = stmt.where(student.columns.first_name == 'Lucas') # Add a where clause

results = conn.execute(stmt).fetchall()

# Loop over the results and print first name, last name, and dob
for result in results:
    print(result.first_name, result.surname, result.dob)


In [ ]:
# Define a list of bad students
bad_students = ['James', 'John', 'Patricia']

stmt = db.select(student)
# Append a where clause to match all the students in_ the bad students list
stmt = stmt.where(student.columns.first_name.in_(bad_students))

# Loop over the ResultProxy and print the first and last names
for result in conn.execute(stmt):
    print(result.first_name, result.surname)


In [ ]:
# Define a list of bad students
bad_students = ['James', 'John', 'Patricia']

stmt = db.select(student)
# Append a where clause to match all the students in_ the bad students list
# that are in course_id 1
stmt = stmt.where(db.and_(student.columns.first_name.in_(bad_students),
                          student.columns.course_id == 1))

# Loop over the ResultProxy and print the first and last names
for result in conn.execute(stmt):
    print(result.first_name, result.surname)

In [ ]:
course_codes = db.Table('course_codes', metadata_obj, autoload=True, autoload_with=engine)

# Build a statement to join student and course_codes tables
stmt = db.select(student, course_codes)
stmt_join = stmt.select_from(
    student.join(course_codes, student.columns.course_id == course_codes.columns.course_id))

results = conn.execute(stmt_join)

# Loop over the results
for result in results:
    print(result.first_name, result.surname, "-", result.course_description)


In [ ]:
course_codes = db.Table('course_codes', metadata_obj, autoload=True, autoload_with=engine)

# Build a statement to join student and course_codes tables
stmt = db.select(student, course_codes)
stmt_join = stmt.select_from(student.join(course_codes,
                                          db.and_(student.columns.course_id == course_codes.columns.course_id,
                                                  course_codes.columns.course_description.like("D%"))))

results = conn.execute(stmt_join)

# Loop over the results
for result in results:
    print(result.first_name, result.surname, "-", result.course_description)


# Extract ERD

### Requirements
Might require python-graphviz `conda install -c conda-forge python-graphviz` and [sqlalchemy_schemadisplay](https://github.com/fschulze/sqlalchemy_schemadisplay) `conda install -c conda-forge sqlalchemy_schemadisplay)`

If not using Conda you can get graphviz [here ](https://graphviz.org/download/). Make sure to add it to the system path during the installation.

### Code

In [18]:
# pip install sqlalchemy-schemadisplay
from sqlalchemy_schemadisplay import create_schema_graph

Save ERD to an external file

In [19]:
from sqlalchemy_schemadisplay import create_schema_graph

graph = create_schema_graph(engine=engine,
                            metadata=metadata_obj,
                            rankdir='LR') # Left to right
graph.write_png('erd.png')

Show ERD on Jupyter

In [ ]:
from PIL import Image
from IPython.display import display

img = Image.open('erd.png')
display(img)